# Nondeterministic Actions: Solving Tic-Tac-Toe with AND-OR-Tree Search

## Introduction 
 
Multiplayer games can be implemented as:
1. __Nondeterministic actions:__ The opponent is seen as part of an environment with nondeterministic actions. Non-determinism is the result of the unknown opponent's moves. 
2. Optimal Decisions: Minimax search (search complete game tree) and alpha-beta pruning.
3. Heuristic Alpha-Beta Tree Search: Cut off tree search and use heuristic to estimate state value. 
4. Monte Carlo Tree search: Simulate playouts to estimate state value. 

Here we will implement search for Tic-Tac-Toe (see [rules](https://en.wikipedia.org/wiki/Tic-tac-toe)).

We will implement
* __AND-OR-Tree search.__

Each action consists of the move by the player (OR levels in the tree) and all possible (i.e., nondeterministic) responses by the opponents (AND levels). The action therefore results in a set of possible states.

We will search for a __conditional plan__ using AND-OR-Tree search. 

The code for the basic functions used for search are implemented in [tictactoe.py](tictactoe.py).

In [ ]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("tictactoe.py", ".")

In [1]:
from tictactoe import empty_board, actions, result, terminal, utility, other, show_board
from tictactoe import random_player, play 

## Solving Tic-Tac-Toe using AND-OR-Tree Search

The opponent can be modeled as part of the environment. The opponent's action as a response to the agent's 
action makes the environment appear non-deterministic (see AIMA page 125 or the slides). The ADS-OR-Tree algorithm uses the following functions:

* `is_goal(state)`: determine if the current board/state has a winner.
* `is_cycle(path)`: determine if we are in a loop (the tic-tac-toe problem has no loops so I do not implement it).
* `results(state, action)`: returns all states following state + action.


### Cycles

The path is typically used for loop checking. We ignore it here since tic-tac-toe does not have cycles.

### Goals

Reaching the goal means that the agent has successfully finished the task. 

For games, `terminal(s)` checks if `s` is a terminal state, but a terminal state can also indicate that 
the player lost (which indicates a failure). This can determined by the 
`utility(s)` function. Tic-tac-toe also has draws and we need to decide if a draw is considered successfully reaching
the goal or a failure.

### Non-deterministic Results

Since the opponent and her moves are part of the environment, `results()` returns for a state $\times$ action combination a set of new states that reflects all possible responses by the opponent.

In [ ]:
def results(state, action, player='x'):
    """All legal outcomes: our move, then every opponent reply if game continues."""
    if terminal(state):
        raise ValueError("Cannot move from a terminal board")
    if action not in actions(state):
        raise ValueError(f"Illegal action: {action}")

    after_move = result(state, player, action)
    # A win/draw ends the game immediately; opponent must NOT move afterwards.
    if terminal(after_move):
        return [after_move]

    opponent = other(player)
    return [result(after_move, opponent, a) for a in actions(after_move)]

show_board(empty_board())
print("Legal replies when X plays the center:")
results(empty_board(), 4)


### Recursive AND-OR Search (DFS)

In [ ]:
DEBUG = 1
COUNT = 0

def and_or_search(state, player='x', draw_is_win=True):
    """Find a conditional plan guaranteeing win, or non-loss if draw_is_win."""
    global COUNT
    COUNT = 0
    plan = or_search(state, player, draw_is_win)
    if DEBUG >= 1:
        print(f"Number of nodes searched: {COUNT}")
    return plan

def or_search(state, player, draw_is_win):
    """OR node: choose one of our legal moves."""
    global COUNT
    COUNT += 1
    u = utility(state, player)
    if u == 1:
        return 'win'
    if u == 0:
        return 'draw' if draw_is_win else None
    if u == -1:
        return None

    for action in actions(state):
        plan = and_search(results(state, action, player), player, draw_is_win)
        if plan is not None:
            return [action, plan]
    return None

def and_search(states, player, draw_is_win):
    """AND node: ALL possible opponent replies must lead to a goal."""
    global COUNT
    COUNT += 1
    plans = []
    for s in states:
        next_plan = or_search(s, player, draw_is_win)
        if next_plan is None:
            return None
        plans.append(['if', s, 'then', next_plan])
    return plans


And-or search looks for a subtree (i.e., an action) that guarantees a win and returns a conditional plan for this subtree. If no such subtree exists, no plan is returned. The algorithm only searches a fraction of the game tree since it abandons the current subtree when it finds the first leaf node that is not a win (see `and_search` function).

# Some Tests

## x is about to win

In [ ]:
board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[3] = 'o'
board[4] = 'x'

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

Why does it play 2 and not 8!!!

In [ ]:
print("Win only:")
%time display(and_or_search(board, player = 'x', draw_is_win = False))

## x can draw_is_win if it chooses 7

In [ ]:

board = empty_board() 
board[0] = 'x'
board[1] = 'o'
board[2] = 'x'
#board[3] = 'o'
board[4] = 'o'

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

In [ ]:
print("Win only:")
%time display(and_or_search(board, player = 'x', draw_is_win = False))

## a bad situation

In [ ]:
board = empty_board() 
board[0] = 'o'
board[2] = 'x'
board[8] = 'o'

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

## Empty board: Only a draw_is_win can be guaranteed

In [ ]:
board = empty_board() 

print("Board:")
show_board(board)

print()
print("Win or draw:")
%time display(and_or_search(board, player = 'x', draw_is_win = True))

In [ ]:
print("Win only:")
%time display(and_or_search(board, player = 'x', draw_is_win = False))

## Experiments

### And-Or Tree Search vs. Random

Put AND-OR search into a wrapper agent function. Note that if AND-OR search cannot guarantee a win then
it does not return a plan, nut `None`. In this case, we do not know what the best (i.e., the least "bad") 
move is, so I make the agent play randomly. Other methods that we will learn about later can determine the utility of a move, even if it does not guarantee a win.

In [ ]:
DEBUG = 0

def and_or_player(board, player = 'x'):
    plan = and_or_search(board, player)

    # if there is no plan then we do a random action
    if plan is None: 
        return np.random.choice(actions(board))
    else:
        return plan[0]


print("and-or-search vs. random:")
%time display(play(and_or_player, random_player))

print()
print("random vs. and-or-search")
%time display(play(random_player, and_or_player))

**Note how slow this is!**

My lazy implementation is a _simple reflex agent_ that recreates a conditional plan every time it is asked for an action. ut since And-Or search already produces a complete conditional plan for the empty board, a better implementation would use a _model-based reflex agent_ that store this precomputed plan and just follow it without more planing.

### And-Or Tree Search vs. And-Or Tree Search

In [ ]:
# No randomness -> run only once

%time display(play(and_or_player, and_or_player, N = 1))

### Kiểm thử hồi quy AND–OR
Kiểm thử không cho đối thủ đi sau trạng thái kết thúc, duyệt tất cả nhánh AND và phân biệt hòa/thắng.


In [ ]:
# Regression tests: terminal handling and universal AND semantics
DEBUG = 0
winning_board = ['x', 'x', ' ', 'o', 'o', ' ', ' ', ' ', ' ']
outcomes = results(winning_board, 2, 'x')
assert len(outcomes) == 1 and utility(outcomes[0], 'x') == 1
assert and_or_search(winning_board, 'x', True)[0] == 2
loss_board = ['o', 'o', 'o', 'x', 'x', ' ', ' ', ' ', ' ']
assert and_search([outcomes[0], loss_board], 'x', True) is None
assert and_or_search(outcomes[0], 'x', True) == 'win'
assert and_or_search(empty_board(), 'x', False) is None
assert and_or_search(empty_board(), 'x', True) is not None
print('AND-OR regressions: PASS')
